# Preprocessing of RGI 01 (Alaska) point mass balance data

This notebook builds the stake (point mass balance, PMB) dataset for Alaska used by MassBalanceMachine. Western Canada & USA (RGI 02) is not processed: it is not used in the paper and has no glacier grids.

**Pipeline**
1. Load WGMS point mass balance data and keep US/CA stakes in GTN-G region `01_alaska`.
2. Add extra Alaskan stakes not contained in WGMS (`01_Alaska.csv`).
3. Download OGGM glacier directories and sample topographic variables (`aspect`, `slope`, `topo`) at each stake; drop stakes outside the RGI outline.
4. Merge stakes closer than 10 m (same glacier, year and period).
5. Add the sky-view factor (SVF) from the pre-computed per-glacier grids.
6. Re-assign stake IDs by elevation.
7. Clean measurement dates (shift July starts, reclassify/filter inconsistent winter and annual periods).
8. Save `ALA_wgms_dataset_all.csv` and plot data overviews.

## Setting up:

In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), '../../../../')) # Add root of repo to import MBM

import warnings
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import massbalancemachine as mbm

# Region-specific paths (path_PMB_WGMS_csv)
from papers.Transformer_TF.PMB_sources.RGI_01_02_US_CA.scripts.config_US_CA import *

# Helper functions shared across regions (OGGM, stake QC, SVF, maps)
from papers.Transformer_TF.scripts.oggm import initialize_oggm_glacier_directories, export_oggm_grids, merge_pmb_with_oggm_data
from papers.Transformer_TF.scripts.pmb_processing import rename_stakes_by_elevation, check_point_ids_contain_glacier, remove_close_points, add_rgi_ids_to_df
from papers.Transformer_TF.scripts.pmb_processing import add_svf_from_rgi_zarr, plot_missing_svf_for_all_glaciers, add_svf_nearest_valid
from papers.Transformer_TF.scripts.plotting.folium import plot_stakes_folium
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Call config file (data paths, seed, etc.)
cfg = mbm.USCANADAConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

## Pre-processing:

### Load WGMS data:
- `mass_balance_point.csv`: point mass balance measurements (one row per stake and period).
- `glacier.csv`: glacier metadata, used to map the WGMS `glacier_id` to its RGI v6 ID (`rgi60_ids`) and GTN-G region.

In [ ]:
# Point mass balance measurements from WGMS
df_stakes = pd.read_csv(
    os.path.join(cfg.dataPath, 'WGMS/raw/mass_balance_point.csv'))

# Glacier metadata (WGMS id, RGI60 id, country, GTN-G region)
df_RGIId = pd.read_csv(cfg.dataPath + 'WGMS/raw/glacier.csv')

# Create a mapping dictionary from gtng_region to all countries
# (only displayed to check which countries belong to which region)
country_to_gtng_region_map = df_RGIId.groupby(
    'gtng_region')['country'].unique().to_dict()
country_to_gtng_region_map

In [ ]:
# Create a mapping dictionary from id to rgi60_ids
id_to_rgi_map = dict(zip(df_RGIId['id'], df_RGIId['rgi60_ids']))

# Create a mapping dictionary from id to gtng_region
id_to_gtng_region_map = dict(zip(df_RGIId['id'], df_RGIId['gtng_region']))

# Add the RGIId and gtng_region columns to the stakes using glacier_id
# (glacier_id in df_stakes corresponds to id in df_RGIId)
df_stakes['RGIId'] = df_stakes['glacier_id'].map(id_to_rgi_map)
df_stakes['gtng_region'] = df_stakes['glacier_id'].map(id_to_gtng_region_map)

print('Unique countries in df_stakes:', df_stakes['country'].unique())
print('Unique gtng_regions in df_stakes:', df_stakes['gtng_region'].unique())

# Filter df_stakes to include only rows where country is US or CA
df_stakes_US_CA = df_stakes[df_stakes['country'].isin(
    ['US', 'CA'])].reset_index(drop=True)

print(df_stakes_US_CA.gtng_region.unique())

# Keep Alaska only (RGI 01); Western Canada & USA (02) and Arctic Canada (03/04)
# are not used
df_stakes_US_CA = df_stakes_US_CA[df_stakes_US_CA['gtng_region'] == '01_alaska']

print('Number of measurements:', len(df_stakes_US_CA))
print('Number of glaciers:', len(df_stakes_US_CA.RGIId.unique()))

# Display glacier names with NaN RGIId (no RGI match in WGMS metadata)
display(
    f"Number of rows with NaN RGIId: {df_stakes_US_CA['RGIId'].isna().sum()}")
display(
    df_stakes_US_CA[df_stakes_US_CA['RGIId'].isna()]['glacier_name'].unique())

# drop rows with NaN RGIId
df_stakes_US_CA = df_stakes_US_CA.dropna(subset=['RGIId']).reset_index(
    drop=True)

print('Number of measurements:', len(df_stakes_US_CA))
print('Number of glaciers:', len(df_stakes_US_CA.RGIId.unique()))

# Sanity check: should now be 0
display(
    f"Number of rows with NaN RGIId: {df_stakes_US_CA['RGIId'].isna().sum()}")
display(
    df_stakes_US_CA[df_stakes_US_CA['RGIId'].isna()]['glacier_name'].unique())
# print number of glaciers per gtng_region
print(df_stakes_US_CA.groupby('gtng_region')['RGIId'].nunique())

In [ ]:
# Select and rename columns to the MBM naming convention
df_stakes_US_CA_renamed = df_stakes_US_CA.rename(
    columns={
        'point_id': 'POINT_ID',
        'latitude': 'POINT_LAT',
        'longitude': 'POINT_LON',
        'elevation': 'POINT_ELEVATION',
        'begin_date': 'FROM_DATE',
        'end_date': 'TO_DATE',
        'balance': 'POINT_BALANCE',
        'glacier_name': 'GLACIER',
        'year': 'YEAR',
        'country': 'COUNTRY',
        'balance_code': 'PERIOD',
        'gtng_region': "RGI_REGION"
    })

# Create new POINT_ID column: <glacier>_<year>_<WGMS measurement id>_<country>
df_stakes_US_CA_renamed['POINT_ID'] = (
    df_stakes_US_CA_renamed['GLACIER'] + '_' +
    df_stakes_US_CA_renamed['YEAR'].astype(str) + '_' +
    df_stakes_US_CA_renamed['id'].astype(str) + '_' +
    df_stakes_US_CA_renamed['COUNTRY'])
# Only keep relevant columns in df
df_stakes_US_CA_renamed = df_stakes_US_CA_renamed[[
    'POINT_ID', 'POINT_LAT', 'POINT_LON', 'POINT_ELEVATION', 'FROM_DATE',
    'TO_DATE', 'POINT_BALANCE', 'GLACIER', 'PERIOD', 'RGIId', 'YEAR',
    "RGI_REGION"
]]

# Remove rows with NaN values in POINT_LAT, POINT_LON, and POINT_ELEVATION
df_stakes_US_CA_renamed = df_stakes_US_CA_renamed.dropna(
    subset=['POINT_LAT', 'POINT_LON', 'POINT_ELEVATION'])

# change date format to YYYYMMDD
df_stakes_US_CA_renamed['FROM_DATE'] = df_stakes_US_CA_renamed[
    'FROM_DATE'].astype(str).str.replace('-', '')
df_stakes_US_CA_renamed['TO_DATE'] = df_stakes_US_CA_renamed['TO_DATE'].astype(
    str).str.replace('-', '')

# Add data modification column to keep track of mannual changes
df_stakes_US_CA_renamed['DATA_MODIFICATION'] = ''
df_stakes_US_CA_renamed.head()

print('Number of measurements:', len(df_stakes_US_CA_renamed))

### Add extra stakes data:
Additional Alaskan stakes not available in WGMS are read from `01_Alaska.csv`. Since this file has no RGI IDs, they are assigned by intersecting the stake coordinates with the RGI v6 region 01 outlines (unmatched points get the nearest outline). Summer balances, rows without dates and rows whose period does not span exactly one hydrological year boundary are removed.

In [ ]:
# File uses ';' separator and latin1 encoding
extra_stakes_df = pd.read_csv('01_Alaska.csv', encoding='latin1', sep=';')

df_extra_stakes_ALA = extra_stakes_df.rename(
    columns={
        'id': 'POINT_ID',
        'latitude': 'POINT_LAT',
        'longitude': 'POINT_LON',
        'elevation': 'POINT_ELEVATION',
        'begin_date': 'FROM_DATE',
        'end_date': 'TO_DATE',
        'balance': 'POINT_BALANCE',
        'glacier_name': 'GLACIER',
        'year': 'YEAR',
        'country': 'COUNTRY',
        'balance_code': 'PERIOD'
    })

# Assign RGIIds by intersecting stake coordinates with RGI v6 outlines
path_rgi_outlines = cfg.dataPath + 'RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp'
df_extra_stakes_ALA = add_rgi_ids_to_df(df_extra_stakes_ALA, path_rgi_outlines)
df_extra_stakes_ALA['COUNTRY'] = 'ALA'  # used as suffix in POINT_ID

# Create new POINT_ID column
df_extra_stakes_ALA['POINT_ID'] = (
    df_extra_stakes_ALA['GLACIER'] + '_' +
    df_extra_stakes_ALA['YEAR'].astype(str) + '_' +
    df_extra_stakes_ALA['POINT_ID'].astype(str) + '_' +
    df_extra_stakes_ALA['COUNTRY'])
# Only keep relevant columns in df
df_extra_stakes_ALA = df_extra_stakes_ALA[[
    'POINT_ID',
    'POINT_LAT',
    'POINT_LON',
    'POINT_ELEVATION',
    'FROM_DATE',
    'TO_DATE',
    'POINT_BALANCE',
    'GLACIER',
    'PERIOD',
    'RGIId',
    'YEAR',
]]

# Display glacier names with NaN RGIId
display(
    f"Number of rows with NaN RGIId: {df_extra_stakes_ALA['RGIId'].isna().sum()}"
)
display(df_extra_stakes_ALA[df_extra_stakes_ALA['RGIId'].isna()]
        ['GLACIER'].unique())

# Remove rows with NaN values in POINT_LAT, POINT_LON, and POINT_ELEVATION
df_extra_stakes_ALA = df_extra_stakes_ALA.dropna(
    subset=['POINT_LAT', 'POINT_LON', 'POINT_ELEVATION'])

# Convert dates from DD.MM.YYYY to YYYYMMDD
df_extra_stakes_ALA['FROM_DATE'] = pd.to_datetime(
    df_extra_stakes_ALA['FROM_DATE'], format='%d.%m.%Y',
    errors='coerce').dt.strftime('%Y%m%d')

df_extra_stakes_ALA['TO_DATE'] = pd.to_datetime(
    df_extra_stakes_ALA['TO_DATE'], format='%d.%m.%Y',
    errors='coerce').dt.strftime('%Y%m%d')

# Add data modification column to keep track of mannual changes
df_extra_stakes_ALA['DATA_MODIFICATION'] = ''
# Derive the GTN-G region from the RGIId (RGI60-01.xxxxx -> 01_alaska)
df_extra_stakes_ALA['RGI_REGION'] = df_extra_stakes_ALA['RGIId'].str.extract(
    r'RGI60-(\d+)\.').squeeze().map({
        '01': '01_alaska',
        '02': '02_western_canada_usa'
    })
# drop period == 'summer'
df_extra_stakes_ALA = df_extra_stakes_ALA[df_extra_stakes_ALA['PERIOD'] !=
                                          'summer']

# drop where TO or FROM_DATE are NaN (incl. dates that failed to parse)
df_extra_stakes_ALA = df_extra_stakes_ALA.dropna(
    subset=['FROM_DATE', 'TO_DATE'])

# Check that FROM_DATE year is TO_DATE year - 1
from_years = pd.to_datetime(df_extra_stakes_ALA['FROM_DATE'],
                            errors='coerce').dt.year
to_years = pd.to_datetime(df_extra_stakes_ALA['TO_DATE'],
                          errors='coerce').dt.year
mismatched = df_extra_stakes_ALA[(to_years - from_years) != 1]
if not mismatched.empty:
    print(
        f"⚠️ {len(mismatched)} rows where FROM_DATE year is not TO_DATE year - 1:"
    )
    display(mismatched[['GLACIER', 'YEAR', 'FROM_DATE', 'TO_DATE', 'PERIOD']])
else:
    print("✅ All rows have FROM_DATE year = TO_DATE year - 1")

# Always drop mismatched rows (outside if/else)
df_extra_stakes_ALA = df_extra_stakes_ALA.drop(mismatched.index)
print(
    f"Dropped {len(mismatched)} mismatched rows, {len(df_extra_stakes_ALA)} rows remaining."
)


### Merge WGMS & other sources:

In [ ]:
# merge the two dfs:
df_stakes_US_CA_renamed = pd.concat(
    [df_stakes_US_CA_renamed, df_extra_stakes_ALA])

# reset_index
df_stakes_US_CA_all = df_stakes_US_CA_renamed.reset_index(drop=True)

print('Number of measurements:', len(df_stakes_US_CA_all))
# Check if any entry anywhere is NaN (should be empty)
display(df_stakes_US_CA_all[df_stakes_US_CA_all.isna().any(axis=1)])

print('Unique RGI regions:', df_stakes_US_CA_all['RGI_REGION'].unique())

In [ ]:
# print max, min POINT_LON and POINT_LAT (sanity check of coordinates)
print('Longitude range:', df_stakes_US_CA_all['POINT_LON'].min(), 'to',
      df_stakes_US_CA_all['POINT_LON'].max())

print('Latitude range:', df_stakes_US_CA_all['POINT_LAT'].min(), 'to',
      df_stakes_US_CA_all['POINT_LAT'].max())

## Add OGGM data:
Download the OGGM glacier directories (OGGM v1.6, `elev_bands_w_data` preprocessed level), export their gridded data, and sample the topographic variables `aspect`, `slope` and `topo` (elevation) at the nearest grid cell of each stake. `merge_pmb_with_oggm_data` also flags whether each stake lies within its RGI glacier outline (`within_glacier_shape`).

In [ ]:
# Alaska (RGI region 01):
df_alaska = df_stakes_US_CA_all[df_stakes_US_CA_all['RGI_REGION'] ==
                                '01_alaska']
RGI_NUMBER = '01'
rgi_ids_list = df_alaska['RGIId'].dropna().unique().tolist()

# initialize OGGM glacier directories
gdirs, rgidf = initialize_oggm_glacier_directories(
    cfg,
    rgi_region=RGI_NUMBER,
    rgi_version="62",
    base_url=
    "https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L1-L2_files/2025.6/elev_bands_w_data/",
    log_level='WARNING',
    task_list=None,
    rgi_ids_list=rgi_ids_list)

# Save the OGGM gridded data of these glaciers to disk
export_oggm_grids(cfg, gdirs, rgi_region=RGI_NUMBER, subset_rgis=rgi_ids_list)


# Sample topographic variables at each stake and flag stakes within the glacier outline
df_stakes_topo_alaska = merge_pmb_with_oggm_data(
    df_pmb=df_alaska,
    gdirs=gdirs,
    rgi_region=RGI_NUMBER,
    rgi_version="62",
    variables_of_interest=['aspect', 'slope', 'topo'])

print("Unique years in the dataset:", df_stakes_topo_alaska['YEAR'].unique())

In [ ]:
df_stakes_topo = df_stakes_topo_alaska.copy()

# Restrict to stakes within the RGI glacier outline
df_stakes_topo = df_stakes_topo[df_stakes_topo['within_glacier_shape'] == True]
df_stakes_topo = df_stakes_topo.drop(columns=['within_glacier_shape'])

# Display rows that have any NaN values
display(df_stakes_topo[df_stakes_topo.isna().any(axis=1)])

## Merge close stakes:
Within each glacier, year and period, stakes closer than 10 m to each other are merged into one point carrying the mean `POINT_BALANCE` (see `remove_close_points`).

In [ ]:
# Keep only annual and winter balances (drops summer and WGMS 'index' rows)
df_stakes_topo = df_stakes_topo[df_stakes_topo['PERIOD'].isin(['annual', 'winter'])]

# Merge stakes < 10 m apart, glacier by glacier
df_pmb_topo = pd.DataFrame()
for gl in tqdm(df_stakes_topo.GLACIER.unique(), desc='Merging stakes'):
    print(f'-- {gl.capitalize()}:')
    df_gl = df_stakes_topo[df_stakes_topo.GLACIER == gl]
    df_gl_cleaned = remove_close_points(df_gl)
    df_pmb_topo = pd.concat([df_pmb_topo, df_gl_cleaned])
# Drop temporary projected coordinates added by remove_close_points
df_pmb_topo.drop(['x', 'y'], axis=1, inplace=True)
df_pmb_topo.reset_index(inplace=True, drop=True)

In [ ]:
# Number of stakes per region after merging
df_pmb_topo.groupby('RGI_REGION').POINT_ID.count()

## Add skyview factor:
The sky-view factor (SVF) is sampled at each stake from the pre-computed per-glacier grids (`<RGIId>.zarr` in `xr_masked_grids/`). Stakes falling on a masked (NaN) pixel are first inspected, then filled with the nearest valid SVF pixel.

In [ ]:
# Map RGI region number to GTN-G region name used in RGI_REGION
RGI_TO_GTNG = {'01': '01_alaska'}

# Directory with the per-glacier SVF grids (<RGIId>.zarr) of each region
RGI_REGIONS_CONFIG = {
    "01":
    os.path.join(cfg.dataPath, 'RGI_v6/RGI_01_Alaska/xr_masked_grids/'),
}

df_pmb_topo_svf_list = []

for rgi_id, path_masked_xr in RGI_REGIONS_CONFIG.items():
    # Filter to current region
    gtng_key = RGI_TO_GTNG[rgi_id]
    df_pmb_topo_region = df_pmb_topo[df_pmb_topo['RGI_REGION'] == gtng_key]
    print(
        f"[{rgi_id}] Processing {len(df_pmb_topo_region)} points from {gtng_key}"
    )

    # Nearest-pixel SVF sampling (NaN if the pixel is masked)
    df_pmb_topo_svf_ = add_svf_from_rgi_zarr(
        df_pmb_topo_region,
        path_masked_xr,
        rgi_col="RGIId",
        lon_col="POINT_LON",
        lat_col="POINT_LAT",
        svf_var="svf",
        out_col="svf",
    )
    df_pmb_topo_svf_list.append(df_pmb_topo_svf_)

# Concatenate all regions
df_pmb_topo_svf = pd.concat(df_pmb_topo_svf_list, ignore_index=True)

# Stakes that fell on a masked (NaN) SVF pixel
df_missing = df_pmb_topo_svf[df_pmb_topo_svf["svf"].isna()].copy()
print("Missing SVF points:", len(df_missing))
print("Glaciers affected:", sorted(df_missing["RGIId"].unique()))

In [ ]:
# Plot SVF grids with missing-SVF stakes (red) and valid stakes (white) per glacier.
path_masked_xr = RGI_REGIONS_CONFIG['01']
plot_missing_svf_for_all_glaciers(
    df_with_svf=df_pmb_topo_svf,
    path_masked_xr=path_masked_xr,
    plot_valid_points=True,
    save_dir=
    None  # or e.g. os.path.join(cfg.dataPath, "diagnostics/svf_missing")
)

In [ ]:
# Directory with the per-glacier SVF grids of each region (keyed by RGI_REGION)
paths_masked_xr = {
    RGI_TO_GTNG[rgi]: path
    for rgi, path in RGI_REGIONS_CONFIG.items()
}


dfs_out = []

for region, df_region in df_pmb_topo.groupby("RGI_REGION", sort=False):
    if region not in paths_masked_xr:
        raise ValueError(f"No masked xr path configured for region {region}")

    path_masked_xr = paths_masked_xr[region]

    print(f"\nProcessing region: {region}")
    print(f"Rows: {len(df_region)}")

    # Sample SVF; points on masked pixels are filled with the nearest
    # valid pixel within max_radius grid cells
    df_region_svf = add_svf_nearest_valid(
        df_region.copy(),
        path_masked_xr,
        rgi_col="RGIId",
        lon_col="POINT_LON",
        lat_col="POINT_LAT",
        svf_var="svf",
        out_col="svf",
        max_radius=60,
    )

    dfs_out.append(df_region_svf)

df_pmb_topo_svf_new = pd.concat(dfs_out, ignore_index=True)

# add_svf_nearest_valid drops glaciers without an SVF grid: check none were lost
assert (len(df_pmb_topo) == len(df_pmb_topo_svf_new))
print("Missing SVF points after nearest-valid fill:",
      df_pmb_topo_svf_new["svf"].isna().sum())

## Give new stake ids:
Stake IDs are re-assigned per glacier as `<GLACIER>_<rank>`, where the rank is given by the stake's mean elevation (0 = lowest).

In [ ]:
# Rename POINT_IDs to <GLACIER>_<rank>, ranked by elevation
df_pmb_new_ids = rename_stakes_by_elevation(df_pmb_topo_svf_new)

# Check that every POINT_ID contains its glacier name
check_point_ids_contain_glacier(df_pmb_new_ids)

print('Number of winter and annual samples:', len(df_pmb_new_ids))
print('Number of annual samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'winter']))

# Histogram of mass balance
df_pmb_new_ids['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

## Cleaning of dates:
Measurement periods are harmonised so that they match the hydrological year used by MBM:
1. Keep years after 1951 and drop summer balances.
2. Shift annual/winter periods starting in July to start on August 1st.
3. Reclassify "winter" measurements ending in June or July as annual.
4. Remove annual measurements that do not start in Aug–Dec, end in Jun–Oct, or last more than 460 days.
5. Remove winter measurements that do not start in Aug–Dec, end in Jan–May, or last more than 430 days.

In [ ]:
df_pmb_clean = df_pmb_new_ids.copy()

# Filter to YEAR > 1951
df_pmb_clean = df_pmb_clean[df_pmb_clean['YEAR'] > 1951]

# drop summer period
df_pmb_clean = df_pmb_clean[df_pmb_clean["PERIOD"] != "summer"].copy()

# Ensure YYYYMMDD format
df_pmb_clean["FROM_DATE"] = df_pmb_clean["FROM_DATE"].astype(str).str.zfill(8)
df_pmb_clean["TO_DATE"] = df_pmb_clean["TO_DATE"].astype(str).str.zfill(8)

# Extract months (MM) of start and end of each measurement
df_pmb_clean["MONTH_START"] = df_pmb_clean["FROM_DATE"].str[4:6]
df_pmb_clean["MONTH_END"] = df_pmb_clean["TO_DATE"].str[4:6]

# Compute duration of each measurement period in days
df_pmb_clean["FROM_DATE_dt"] = pd.to_datetime(df_pmb_clean["FROM_DATE"],
                                              format="%Y%m%d",
                                              errors="coerce")
df_pmb_clean["TO_DATE_dt"] = pd.to_datetime(df_pmb_clean["TO_DATE"],
                                            format="%Y%m%d",
                                            errors="coerce")
df_pmb_clean["duration_days"] = (df_pmb_clean["TO_DATE_dt"] -
                                 df_pmb_clean["FROM_DATE_dt"]).dt.days


# Helper: print unique start/end months for winter and annual measurements
def print_months(df, label):
    winter = df[df.PERIOD == "winter"]
    annual = df[df.PERIOD == "annual"]
    print(f"\n{label}")
    print("Winter measurement months:")
    print("  Unique start months:", sorted(winter["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(winter["MONTH_END"].unique()))
    print("\nAnnual measurement months:")
    print("  Unique start months:", sorted(annual["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(annual["MONTH_END"].unique()))


# Helper: count and duration stats for each (start month, end month) combination
def print_combos(df, period, label):
    sub = df[df["PERIOD"] == period].copy()
    combo_counts = (sub.groupby(["MONTH_START", "MONTH_END"]).agg(
        count=("duration_days", "size"),
        mean_days=("duration_days", "mean"),
        min_days=("duration_days", "min"),
        max_days=("duration_days", "max")).reset_index().sort_values(
            ["MONTH_START", "MONTH_END"]))
    combo_counts["mean_days"] = combo_counts["mean_days"].round(0).astype(int)
    print(f"\n{label}")
    print(combo_counts.to_string(index=False))


# --- Before anything ---
print_months(df_pmb_clean, "Before any changes")
print_combos(df_pmb_clean, "annual",
             "Annual: all combinations before shifting")
print_combos(df_pmb_clean, "winter",
             "Winter: all combinations before shifting")

# ======================
# 1) SHIFT JULY STARTS TO AUGUST 1st
# ======================
# July starts are moved to August 1st (start of the hydrological year)
# and the duration is recomputed
for period_type in ["annual", "winter"]:
    mask_july = ((df_pmb_clean["PERIOD"] == period_type) &
                 (df_pmb_clean["MONTH_START"] == "07"))
    n_july = int(mask_july.sum())
    df_pmb_clean.loc[mask_july, "FROM_DATE"] = (
        df_pmb_clean.loc[mask_july, "FROM_DATE_dt"].apply(
            lambda d: d.replace(month=8, day=1).strftime("%Y%m%d")))
    df_pmb_clean.loc[mask_july, "FROM_DATE_dt"] = pd.to_datetime(
        df_pmb_clean.loc[mask_july, "FROM_DATE"], format="%Y%m%d")
    df_pmb_clean.loc[mask_july, "duration_days"] = (
        df_pmb_clean.loc[mask_july, "TO_DATE_dt"] -
        df_pmb_clean.loc[mask_july, "FROM_DATE_dt"]).dt.days
    df_pmb_clean.loc[mask_july, "MONTH_START"] = "08"
    print(
        f"Shifted {n_july} {period_type} measurements from July to August 1st start"
    )

print_months(df_pmb_clean, "After shifting July starts to August")

# ======================
# 2) RECLASSIFY WINTER ENDING IN JUNE/JULY AS ANNUAL
# ======================
# A "winter" balance measured in June/July effectively spans the year
mask_reclassify = ((df_pmb_clean["PERIOD"] == "winter") &
                   (df_pmb_clean["MONTH_END"].isin(["06", "07"])))
n_reclassify = int(mask_reclassify.sum())
df_pmb_clean.loc[mask_reclassify, "PERIOD"] = "annual"
print(f"Reclassified {n_reclassify} winter rows ending in June/July as annual")

# ======================
# 3) FILTER INVALID ANNUAL
# ======================
# Annual: start Aug-Dec, end Jun-Oct, at most 460 days
valid_annual_end = ["06", "07", "08", "09", "10"]
valid_annual_start = ["08", "09", "10", "11", "12"]
mask_invalid_annual = ((df_pmb_clean["PERIOD"] == "annual") & ~(
    (df_pmb_clean["MONTH_END"].isin(valid_annual_end)) &
    (df_pmb_clean["MONTH_START"].isin(valid_annual_start)) &
    (df_pmb_clean["duration_days"] <= 460)))
n_invalid_annual = int(mask_invalid_annual.sum())
print(f"\nInvalid annual rows to remove: {n_invalid_annual}")
print("\nAnnual combinations being removed:")
print(df_pmb_clean[mask_invalid_annual].groupby([
    "MONTH_START", "MONTH_END"
]).agg(count=("duration_days", "size"),
       mean_days=("duration_days", "mean"),
       min_days=("duration_days", "min"),
       max_days=("duration_days", "max")).reset_index().to_string(index=False))
df_pmb_clean = df_pmb_clean.loc[~mask_invalid_annual].copy()

# ======================
# 4) FILTER INVALID WINTER
# ======================
# Winter: start Aug-Dec, end Jan-May, at most 430 days
valid_winter_end = ["01", "02", "03", "04", "05"]
valid_winter_start = ["08", "09", "10", "11", "12"]
mask_invalid_winter = ((df_pmb_clean["PERIOD"] == "winter") & ~(
    (df_pmb_clean["MONTH_END"].isin(valid_winter_end)) &
    (df_pmb_clean["MONTH_START"].isin(valid_winter_start)) &
    (df_pmb_clean["duration_days"] <= 430)))
n_invalid_winter = int(mask_invalid_winter.sum())
print(f"\nInvalid winter rows to remove: {n_invalid_winter}")
print("\nWinter combinations being removed:")
print(df_pmb_clean[mask_invalid_winter].groupby([
    "MONTH_START", "MONTH_END"
]).agg(count=("duration_days", "size"),
       mean_days=("duration_days", "mean"),
       min_days=("duration_days", "min"),
       max_days=("duration_days", "max")).reset_index().to_string(index=False))
df_pmb_clean = df_pmb_clean.loc[~mask_invalid_winter].copy()

print_months(df_pmb_clean, "After all filtering")

## Save final dataframe:
Drop helper columns and save the Alaska CSV to `cfg.dataPath/path_PMB_WGMS_csv`.

In [ ]:
# Helper / metadata columns not needed in the final dataset
columns_to_drop = [
    'begin_date_unc', 'end_date_unc', 'RGI_REGION', 'MONTH_START', 'MONTH_END',
    'FROM_DATE_dt', 'TO_DATE_dt', 'duration_days', 'DATA_MODIFICATION'
]

# Alaska (RGI 01)
df_mb_alaska = df_pmb_clean[df_pmb_clean.RGI_REGION == '01_alaska'].drop(
    columns=columns_to_drop, errors='ignore')
df_mb_alaska.reset_index(inplace=True, drop=True)
df_mb_alaska.to_csv(os.path.join(cfg.dataPath, path_PMB_WGMS_csv,
                                 'ALA_wgms_dataset_all.csv'),
                    index=False)

# Histogram of mass balance
df_pmb_clean['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

In [ ]:
# Interactive map of all cleaned stakes, one layer per glacier
m = plot_stakes_folium(df_pmb_clean, color_map=None, layer_control=True)
m

In [ ]:
print('Number of winter and annual samples:', len(df_pmb_clean))
print('Number of annual samples:',
      len(df_pmb_clean[df_pmb_clean.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_clean[df_pmb_clean.PERIOD == 'winter']))

# Number of measurements per year (stacked annual/winter):
fig, axs = plt.subplots(2, 1, figsize=(20, 15))
ax = axs.flatten()[0]
df_pmb_clean.groupby(['YEAR', 'PERIOD']).size().unstack().plot(
    kind='bar',
    stacked=True,
    color=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
    ax=ax)
ax.set_title('Number of measurements per year for all glaciers')

# Total number of measurements per glacier
ax = axs.flatten()[1]
num_gl = df_pmb_clean.groupby(['GLACIER']).size().sort_values()
num_gl.plot(kind='bar', ax=ax)
ax.set_title('Number of total measurements per glacier since 1951')
plt.tight_layout()